# DoS Stress-Testing & Auto-Mitigation — Final Demo

**Real backend:** Qwen2.5-0.5B-Instruct, vLLM, Colab T4.  
**Pipeline:** local Locust traffic → mitigation gateway → real vLLM → saved measurements.

This notebook consolidates the useful setup/integration steps and preserves the recorded smoke-test results using the original `testing.ipynb` and `integration.ipynb`.



## A1 — Locate the project and choose what to run

An existing project folder is reused without deleting it. Otherwise an uploaded project ZIP is extracted; GitHub cloning is the last fallback. Keep the ports fixed for a new run. Both mitigation modes use the same gateway port.

In [ ]:
from pathlib import Path
import os, sys, json, subprocess, zipfile

RUN_LIVE_DEMO = False
PROFILES = ["flood"]  # all four: ["normal", "spike", "flood", "low_slow"]
MODEL = "Qwen/Qwen2.5-0.5B-Instruct"
BACKEND_URL = "http://127.0.0.1:8000"
GATEWAY_PORT = 8090
MAX_SEQS = 8
LIVE_RUNS = {}

candidates = [Path.cwd(), Path.cwd().parent,
              Path("/content/DoS-Stress-Testing"),
              Path("/content/DoS-Stress-Testing-main"),
              Path("/content/demo_source/DoS-Stress-Testing-main")]
REPO = next((p for p in candidates if (p / "gateway/app.py").is_file()), None)
if REPO is None:
    base = Path("/content") if Path("/content").exists() else Path.cwd()
    archives = sorted(base.glob("DoS-Stress-Testing*.zip"),
                      key=lambda p: p.stat().st_mtime, reverse=True)
    if archives:
        destination = base / "demo_source"
        destination.mkdir(exist_ok=True)
        with zipfile.ZipFile(archives[0]) as z:
            for member in z.infolist():
                if not (destination / member.filename).resolve().is_relative_to(destination.resolve()):
                    raise ValueError("Unsafe ZIP path")
            z.extractall(destination)
        REPO = next((p.parent.parent for p in destination.glob("*/gateway/app.py")), None)
    if REPO is None:
        REPO = base / "DoS-Stress-Testing"
        subprocess.run(["git", "clone", "https://github.com/mena-04/DoS-Stress-Testing.git",
                        str(REPO)], check=True)
REPO = REPO.resolve()
os.chdir(REPO)
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))
RESULTS = REPO / "results"
RESULTS.mkdir(exist_ok=True)
(REPO / "loadgen").mkdir(exist_ok=True)
print("Project:", REPO)
print("Live replay:", RUN_LIVE_DEMO)
print("Results:", RESULTS)

## A2 — Existing recorded result, not a new run

The uploaded CSV was transcribed from the successful console outputs in `integration.ipynb`. It is retained as historical evidence, not presented as newly recomputed request-level statistics. The successful OFF output contains **3 legitimate requests** and the ON output contains **15**; these are small-sample demo observations, not reliable production p95 estimates.

The old smoke generator is closed-loop: the same settings do not produce identical arrival timestamps or request counts after mitigation. Failed connection/502 runs elsewhere in the original notebook are debugging runs, not part of this comparison.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

saved = RESULTS / "flood_before_after.csv"
if saved.is_file():
    recorded = pd.read_csv(saved)
    display(recorded)
    fig, ax = plt.subplots(figsize=(6, 4))
    ax.bar(recorded["mode"], recorded["legit_p95_s"])
    ax.set(xlabel="Mitigation", ylabel="Legitimate p95 latency (s)",
           title="Recorded smoke-test comparison — small sample")
    fig.tight_layout()
    fig.savefig(RESULTS / "recorded_flood_p95.png", dpi=200, bbox_inches="tight")
    plt.show()
    plt.close(fig)
else:
    print("The historical CSV is not present in this project copy. No values were invented.")

### Original successful console outputs

These are copied verbatim from the uploaded integration notebook, not executed results of the new cells.

**OFF — integration.ipynb, original code-cell index 20**
```text
=== legit (3 requests) ===
  success rate   100.0%
  rejection rate 0.0%
  p50 8.867s  p95 9.546s  max 9.546s
  statuses       {'200': 3}

=== attacker (24 requests) ===
  success rate   100.0%
  rejection rate 0.0%
  p50 8.731s  p95 10.355s  max 10.355s
  statuses       {'200': 24}
```
**ON — integration.ipynb, original code-cell index 18**
```text
=== legit (15 requests) ===
  success rate   100.0%
  rejection rate 0.0%
  p50 0.894s  p95 1.323s  max 1.323s
  statuses       {'200': 15}

=== attacker (16 requests) ===
  success rate   50.0%
  rejection rate 50.0%
  p50 11.630s  p95 12.113s  max 12.113s
  reasons        {'client_concurrency': 8}
  statuses       {'429': 8, '200': 8}
```
The saved queue-summary output in that notebook reports a maximum observed upstream waiting count of **1 in each run**, and ON gateway in-flight maximum **5**. Those maxima cannot reconstruct a time series. A correct queue plot may therefore show similar peaks; do not change values to make the picture more dramatic.

## A3 — Queue-depth plotting code

Reads `ts`, `upstream_waiting` and `upstream_stale` from the gateway's existing samples. **No second sampler is started.** Stale/unknown queue readings appear as gaps, not zeroes. The CSV export keeps the source paths and timestamps.

New runs use their saved load-start time. For historical runs without timing metadata, the x-axis is elapsed time from each file's first sample; attack windows are not guessed.

In [ ]:
"""Plot actual gateway samples. Missing or stale readings never become zero."""
from pathlib import Path
import json
import pandas as pd
import matplotlib.pyplot as plt


def plot_queue_depth(paths, output_dir):
    """paths is e.g. {'OFF': Path(...jsonl), 'ON': Path(...jsonl)}."""
    output_dir = Path(output_dir)
    missing = [str(p) for p in paths.values() if not Path(p).is_file()]
    if missing:
        print("No queue chart generated: the raw files below are missing.")
        for p in missing:
            print("  ", p)
        print("Use the original Colab runtime, or upload these JSONL files from it.")
        print("The supplied ZIP contains the summary/PNG, but not these raw logs.")
        return None

    frames, notes = [], []
    for mode, path in paths.items():
        path = Path(path)
        # Ignore only an incomplete trailing line, as a live logger may still
        # be writing it; never silently discard an interior corrupt record.
        lines = path.read_text().splitlines()
        records = []
        for i, line in enumerate(lines):
            if not line.strip():
                continue
            try:
                records.append(json.loads(line))
            except json.JSONDecodeError:
                if i != len(lines) - 1:
                    raise ValueError(f"Malformed JSON inside {path}, line {i+1}")
                print(f"Ignoring incomplete trailing line in {path.name}.")
        frame = pd.DataFrame(records)
        if frame.empty or not {"ts", "upstream_waiting"}.issubset(frame.columns):
            print(f"No usable timestamp/queue records in {path}")
            return None
        frame["ts"] = pd.to_numeric(frame["ts"], errors="coerce")
        frame = frame.dropna(subset=["ts"]).sort_values("ts").copy()
        if frame.empty:
            print(f"No usable timestamps in {path}")
            return None

        # New runs have exact load-start timestamps. Old runs do not; their
        # x-axis honestly uses each file's first saved sample instead.
        load_meta = path.parent / "load_profile.json"
        metadata = json.loads(load_meta.read_text()) if load_meta.exists() else {}
        start_ts = metadata.get("start_ts", frame["ts"].iloc[0])
        if "start_ts" in metadata:
            frame = frame[frame["ts"] >= start_ts].copy()
        if frame.empty:
            print(f"No samples after load start in {path}")
            return None
        frame["elapsed_s"] = frame["ts"] - start_ts
        frame["waiting"] = pd.to_numeric(frame["upstream_waiting"], errors="coerce")
        if "upstream_stale" in frame:
            stale = frame["upstream_stale"].map(
                lambda x: True if pd.isna(x) else str(x).lower() in {"true", "1", "1.0"}
            )
            frame.loc[stale, "waiting"] = float("nan")
        else:
            stale = pd.Series(False, index=frame.index)
            print(f"{mode}: this file has no staleness flag; freshness cannot be checked.")
        frame.loc[frame["waiting"] < 0, "waiting"] = float("nan")
        frame["mode"] = mode
        frame["source"] = str(path)
        frame["stale"] = stale
        if frame["waiting"].notna().sum() == 0:
            print(f"{mode}: no fresh queue readings. Not making a misleading comparison.")
            return None
        notes.append({"mode": mode, "samples": len(frame),
                      "fresh_samples": int(frame["waiting"].notna().sum()),
                      "stale_fraction": float(stale.mean()),
                      "max_observed_queue": float(frame["waiting"].max()),
                      "time_origin": "load start" if "start_ts" in metadata else "first saved sample"})
        frames.append(frame)

    output_dir.mkdir(parents=True, exist_ok=True)
    fig, ax = plt.subplots(figsize=(8, 4))
    for frame in frames:
        ax.plot(frame["elapsed_s"], frame["waiting"],
                drawstyle="steps-post", label=frame["mode"].iloc[0], linewidth=1.6)
    ax.set(xlabel="Seconds from each run's origin (see summary)",
           ylabel="vLLM waiting requests",
           title="Observed backend queue depth: mitigation OFF vs ON")
    ax.set_ylim(bottom=0)
    ax.legend()
    ax.grid(alpha=0.25)
    fig.tight_layout()
    png = output_dir / "queue_depth_off_vs_on.png"
    csv = output_dir / "queue_depth_samples.csv"
    fig.savefig(png, dpi=200, bbox_inches="tight")
    pd.concat(frames, ignore_index=True)[
        ["mode", "ts", "elapsed_s", "waiting", "stale", "source"]
    ].to_csv(csv, index=False)
    pd.DataFrame(notes).to_csv(output_dir / "queue_depth_summary.csv", index=False)
    print(pd.DataFrame(notes).to_string(index=False))
    print("Saved:", png, "and", csv)
    plt.show()
    plt.close(fig)
    return png

## A4 — Make the queue chart from your existing Colab logs

These are the run IDs used in your successful old notebook. The cell searches both the repo's `runs/` folder and `/content/runs/`. It does not claim that a reused run directory is clean: if you appended failed attempts to the same run ID, choose a clean directory or review the data before presenting it.

When files are present, this creates `results/queue_depth_off_vs_on.png`, `queue_depth_samples.csv`, and `queue_depth_summary.csv`. When absent, it prints the expected locations and makes no chart.

In [ ]:
OFF_RUN = "flood-off-8"
ON_RUN = "flood-on-fresh"

def find_samples(run_id):
    candidates = [REPO / "runs" / run_id / "gateway_samples.jsonl",
                  Path("/content/runs") / run_id / "gateway_samples.jsonl"]
    return next((p for p in candidates if p.is_file()), candidates[0])

QUEUE_PATHS = {"OFF": find_samples(OFF_RUN), "ON": find_samples(ON_RUN)}
queue_png = plot_queue_depth(QUEUE_PATHS, RESULTS)
if queue_png is None:
    available = sorted((REPO / "runs").glob("*/gateway_samples.jsonl"))
    available += sorted(Path("/content/runs").glob("*/gateway_samples.jsonl"))
    print("Available sample files:")
    for path in available:
        print(path)

# B — Optional new live demo

Skip this section when you only need the existing chart. It deliberately uses **real vLLM**, not `gateway.fake_vllm`. New logs get unique run IDs and are not mixed with the historical CSV.

The original gateway implementation is unchanged. Process management is consolidated: one gateway at a time, a fixed port, readiness and real-inference checks, captured logs, and bounded shutdown waits.

## B1 — Dependencies, only for live replay

The vLLM installation route and version come from the successful integration notebook (`vllm 0.29.0`, CUDA 13.0 wheels). A working CLI is reused. The install cell is skipped entirely unless A1 enables live replay. It also installs the project's `dev` extra, because the original notebook's pytest attempt was missing `pytest_asyncio`.

Dependency availability and GPU startup must still be verified in your actual Colab runtime; no fresh GPU run has been executed to create this notebook.

In [ ]:
import shutil

if RUN_LIVE_DEMO:
    subprocess.run(["nvidia-smi"], check=True)
    cli = shutil.which("vllm")
    usable = False
    if cli:
        probe = subprocess.run([cli, "--version"], capture_output=True, text=True, timeout=90)
        usable = probe.returncode == 0
        print(probe.stdout[-500:] if usable else "Existing vLLM CLI needs repair.")
    if not usable:
        subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y",
                        "torch", "torchvision", "torchaudio", "vllm"], check=True)
        subprocess.run([sys.executable, "-m", "pip", "install", "uv"], check=True)
        subprocess.run(["uv", "pip", "install", "--system", "vllm==0.29.0",
                        "--torch-backend=cu130"], check=True)
        subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "torchaudio"], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-e", str(REPO) + "[dev]",
                    "locust", "pandas", "matplotlib"], check=True)
    subprocess.run(["vllm", "--version"], check=True)
else:
    print("Live replay disabled: no installs, no GPU changes.")

## B2 — Process and experiment helpers

The gateway copies retain the uploaded YAML's thresholds; their capacity is aligned with `MAX_SEQS=8`, with two reserved cheap slots. The uploaded `ratelimit_queue.yaml` already uses 8 / 8 / 2. The original files are not overwritten.

Gateway health is checked **and a completion is requested through it** before any load starts. A new run does not start while the backend is still serving the previous run.

In [ ]:
"""Notebook helpers: one managed gateway, real-backend checks, bounded waits.

REPO, MODEL, BACKEND_URL, GATEWAY_PORT and MAX_SEQS are set by the notebook.
The original gateway implementation is not modified.
"""
import json
import os
import socket
import subprocess
import sys
import time
from pathlib import Path
from uuid import uuid4

import requests
import yaml

HTTP = requests.Session()
HTTP.trust_env = False
_gateway = None


def checked_json(url, **kwargs):
    response = HTTP.get(url, timeout=5, **kwargs)
    response.raise_for_status()
    return response.json()


def wait_ready(process, base_url, log_path, seconds=180, expected_mode=None):
    deadline = time.monotonic() + seconds
    while time.monotonic() < deadline:
        if process is not None and process.poll() is not None:
            raise RuntimeError(f"Process exited ({process.returncode}).\n" +
                               Path(log_path).read_text(errors="replace")[-7000:])
        try:
            response = HTTP.get(base_url + "/health", timeout=2)
            if response.status_code == 200:
                if expected_mode is not None and response.json().get("mode") != expected_mode:
                    raise RuntimeError("Another gateway is answering on the selected port.")
                return
        except (requests.RequestException, ValueError):
            pass
        time.sleep(1)
    tail = Path(log_path).read_text(errors="replace")[-7000:] if Path(log_path).exists() else ""
    raise TimeoutError(f"Not ready after {seconds}s. Do not run the load test.\n{tail}")


def verify_real_backend():
    response = HTTP.get(BACKEND_URL + "/health", timeout=5)
    response.raise_for_status()
    # The repository's fake backend has no /version route.
    version = checked_json(BACKEND_URL + "/version")
    models = checked_json(BACKEND_URL + "/v1/models")
    ids = [m.get("id") for m in models.get("data", [])]
    if MODEL not in ids:
        raise RuntimeError(f"Expected model {MODEL}; found {ids}")
    return {"kind": "real_vllm", "version_response": version, "model": MODEL}


def wait_idle(seconds=120):
    from gateway.backpressure import parse_metric
    deadline = time.monotonic() + seconds
    while time.monotonic() < deadline:
        response = HTTP.get(BACKEND_URL + "/metrics", timeout=5)
        response.raise_for_status()
        running = parse_metric(response.text, "vllm:num_requests_running")
        waiting = parse_metric(response.text, "vllm:num_requests_waiting")
        if running == 0 and waiting == 0:
            time.sleep(1)
            return
        time.sleep(1)
    raise RuntimeError("Backend still has work queued. No new experiment was started.")


def stop_gateway():
    global _gateway
    if _gateway is None or _gateway.poll() is not None:
        _gateway = None
        return
    _gateway.terminate()
    try:
        _gateway.wait(timeout=15)
    except subprocess.TimeoutExpired:
        _gateway.kill()
        _gateway.wait(timeout=5)
        print("Gateway required force-stop; its final buffered records may be incomplete.")
    _gateway = None


def prepare_configs():
    paths = {}
    for mode, original in [("off", "off.yaml"), ("on", "ratelimit_queue.yaml")]:
        config = yaml.safe_load((REPO / "configs" / original).read_text())
        backpressure = config.setdefault("backpressure", {})
        backpressure.update(global_max_inflight=MAX_SEQS,
                            upstream_max_num_seqs=MAX_SEQS,
                            reserved_cheap_slots=2)
        config.setdefault("upstream", {})["base_url"] = BACKEND_URL
        # These copies preserve the other thresholds in the uploaded config.
        path = REPO / "configs" / f"demo_{mode}.yaml"
        path.write_text(yaml.safe_dump(config, sort_keys=False))
        paths[mode] = path
    return paths


def run_locust_profile(profile, mode, pair_id):
    """Run one local profile, saving raw client + gateway logs in one folder."""
    global _gateway
    if mode not in {"off", "on"}:
        raise ValueError("mode must be off or on")
    stop_gateway()
    backend = verify_real_backend()
    wait_idle()
    # Refuse collisions rather than silently talking to an unrelated process.
    with socket.socket() as sock:
        if sock.connect_ex(("127.0.0.1", GATEWAY_PORT)) == 0:
            raise RuntimeError(f"Port {GATEWAY_PORT} is occupied. Stop the older gateway first; "
                               "do not start several gateway copies.")
    run_id = f"{pair_id}-{profile}-{mode}"
    run_dir = REPO / "runs" / run_id
    run_dir.mkdir(parents=True, exist_ok=False)
    configs = prepare_configs()
    target = f"http://127.0.0.1:{GATEWAY_PORT}"
    env = os.environ.copy()
    for key in ("GATEWAY_MODE", "GATEWAY_UPSTREAM", "GATEWAY_RUN_ID"):
        env.pop(key, None)
    env["PYTHONUNBUFFERED"] = "1"
    command = [sys.executable, "-u", "-m", "gateway", "--config", str(configs[mode]),
               "--upstream", BACKEND_URL, "--host", "127.0.0.1", "--port", str(GATEWAY_PORT),
               "--run-id", run_id, "--log-dir", str(REPO / "runs")]
    gateway_log = run_dir / "gateway_process.log"
    with gateway_log.open("w") as log:
        _gateway = subprocess.Popen(command, cwd=REPO, env=env, stdout=log,
                                    stderr=subprocess.STDOUT, start_new_session=True)
    try:
        expected = "off" if mode == "off" else "ratelimit_queue"
        wait_ready(_gateway, target, gateway_log, expected_mode=expected)
        # A health response alone does not prove forwarding works.
        test = HTTP.post(target + "/v1/chat/completions", json={
            "model": MODEL, "messages": [{"role": "user", "content": "Say hello."}],
            "max_tokens": 8, "temperature": 0,
        }, headers={"X-Client-ID": "warmup", "X-Traffic-Label": "warmup",
                    "X-Request-ID": uuid4().hex}, timeout=(5, 120))
        if test.status_code != 200 or not test.json().get("choices"):
            raise RuntimeError(f"Gateway forwarding failed: {test.status_code} {test.text[:300]}")
        wait_idle()
        metadata = {"run_id": run_id, "profile": profile, "mode": mode,
                    "backend": backend, "max_num_seqs": MAX_SEQS,
                    "gateway_port": GATEWAY_PORT, "backend_url": BACKEND_URL,
                    "note": "Same user schedule/seed, not identical arrival timestamps.",
                    "created_at": time.time()}
        (run_dir / "demo_manifest.json").write_text(json.dumps(metadata, indent=2))
        env.update(DOS_PROFILE=profile, DOS_RUN_ID=run_id, DOS_RUN_DIR=str(run_dir), DOS_SEED="42")
        cmd = [sys.executable, "-m", "locust", "-f", str(REPO / "loadgen" / "locustfile.py"),
               "--headless", "--host", target, "--csv", str(run_dir / "locust"),
               "--csv-full-history", "--stop-timeout", "125", "--exit-code-on-error", "0"]
        print(f"Running {profile} / {mode.upper()} -> {run_dir.name}", flush=True)
        locust_log = run_dir / "locust.log"
        with locust_log.open("w") as log:
            load = subprocess.Popen(cmd, cwd=REPO, env=env, stdout=log,
                                    stderr=subprocess.STDOUT, start_new_session=True)
        deadline = time.monotonic() + 210
        try:
            while load.poll() is None:
                if _gateway.poll() is not None:
                    raise RuntimeError("Gateway stopped during the run.\n" +
                                       gateway_log.read_text(errors="replace")[-4000:])
                if time.monotonic() > deadline:
                    raise TimeoutError("Load test exceeded its bounded run time.")
                time.sleep(1)
        finally:
            if load.poll() is None:
                load.terminate()
                try:
                    load.wait(timeout=5)
                except subprocess.TimeoutExpired:
                    load.kill()
                    load.wait(timeout=5)
        if load.returncode != 0:
            raise RuntimeError(locust_log.read_text(errors="replace")[-7000:])
        verify_real_backend()
        wait_idle()
        time.sleep(1)  # allow the gateway's buffered log flusher to finish
        print("Finished:", run_dir)
        return run_dir
    finally:
        stop_gateway()

## B3 — Start or verify the real model server

Same serving settings as the successful source notebook. An already running, matching vLLM service is reused. The helper will not silently replace a different service on port 8000. Keep the serving flags identical across OFF/ON conditions.

Prefix caching remains at the original server default. The new Locust profiles vary the beginning of prompts deterministically and log actual generated token counts; these new measurements should not be silently substituted for the old smoke results.

In [ ]:
if RUN_LIVE_DEMO:
    backend_alive = False
    try:
        backend_alive = HTTP.get(BACKEND_URL + "/health", timeout=3).status_code == 200
    except requests.RequestException:
        pass
    if not backend_alive:
        with socket.socket() as sock:
            if sock.connect_ex(("127.0.0.1", 8000)) == 0:
                raise RuntimeError("Port 8000 is occupied by another service. Check it before proceeding.")
        vllm_log_path = RESULTS / "vllm_demo.log"
        with vllm_log_path.open("w") as log:
            server = subprocess.Popen([
                "vllm", "serve", MODEL, "--dtype", "half", "--max-model-len", "2048",
                "--gpu-memory-utilization", "0.85", "--max-num-seqs", str(MAX_SEQS),
                "--host", "127.0.0.1", "--port", "8000",
            ], cwd=REPO, stdout=log, stderr=subprocess.STDOUT, start_new_session=True)
        wait_ready(server, BACKEND_URL, vllm_log_path, seconds=600)
    print("REAL BACKEND:", verify_real_backend())
    response = HTTP.post(BACKEND_URL + "/v1/chat/completions", json={
        "model": MODEL,
        "messages": [{"role": "user", "content": "Explain AI inference in one sentence."}],
        "max_tokens": 32, "temperature": 0,
    }, timeout=(5, 120))
    response.raise_for_status()
    print(response.json()["choices"][0]["message"]["content"])
    print("Usage:", response.json().get("usage"))
    metadata = {"backend": verify_real_backend(), "max_num_seqs": MAX_SEQS,
                "gateway_port": GATEWAY_PORT, "backend_url": BACKEND_URL,
                "max_model_len": 2048, "dtype": "half", "gpu_memory_utilization": 0.85}
    metadata["nvidia_smi"] = subprocess.check_output(["nvidia-smi"], text=True)
    metadata["pip_freeze"] = subprocess.check_output([sys.executable, "-m", "pip", "freeze"], text=True)
    (RESULTS / "demo_environment.json").write_text(json.dumps(metadata, indent=2))
else:
    print("Skipped real-server launch.")

## B4 — All four Locust profiles in one file

Restored from the earlier conversation, not found in the uploaded testing notebook. **Integration edits:** fixed four legitimate users, one selectable shape, per-client and request IDs, `legit`/`attacker` labels for logging, request-level records, finite timeouts, deterministic varying prompt prefixes, and no retries.

| Profile | Normal stage | Attack stage | Recovery |
|---|---|---|---|
| `normal` | 4 legitimate users for 30 seconds | None | Not applicable |
| `spike` | 0–10 s | 10–20 s: 4 legitimate + 20 attacker users | 20–30 s |
| `flood` | 0–10 s | 10–40 s: 4 legitimate + 28 attacker users | 40–50 s |
| `low_slow` | 0–10 s | 10–40 s: 4 legitimate + 4 slower, expensive attacker users | 40–50 s |

Locust uses **closed-loop user pacing** here, so the offered request rate can change as responses speed up or slow down. Report this limitation. Neither the profile nor low-and-slow overload is guaranteed to produce a specific outcome on every machine.

Sources for the added API usage: Locust's official [custom shapes](https://docs.locust.io/en/stable/custom-load-shape.html), [User API](https://docs.locust.io/en/stable/api.html), and [request-event/context documentation](https://docs.locust.io/en/stable/extending-locust.html).

In [ ]:
%%writefile loadgen/locustfile.py
"""Four local-only profiles restored from the earlier Colab/chat examples.

Integration changes: four fixed legitimate users, per-client/request IDs,
finite timeouts, no retries, explicit status logging, and one selectable shape.
This is a closed-loop virtual-user test, not a fixed-arrival-rate benchmark.
"""
import itertools
import json
import os
import random
import time
from pathlib import Path
from urllib.parse import urlparse

from locust import HttpUser, LoadTestShape, between, events, task

PROFILE = os.environ.get("DOS_PROFILE", "normal")
RUN_ID = os.environ.get("DOS_RUN_ID", "demo")
RUN_DIR = Path(os.environ.get("DOS_RUN_DIR", "results/locust-demo"))
SEED = int(os.environ.get("DOS_SEED", "42"))
MODEL = "Qwen/Qwen2.5-0.5B-Instruct"
TIMELINES = {
    "normal": {"end": 30, "attack_start": None, "attack_end": None, "total_users": 4},
    "spike": {"end": 30, "attack_start": 10, "attack_end": 20, "total_users": 24},
    "flood": {"end": 50, "attack_start": 10, "attack_end": 40, "total_users": 32},
    "low_slow": {"end": 50, "attack_start": 10, "attack_end": 40, "total_users": 8},
}
if PROFILE not in TIMELINES:
    raise ValueError(f"Unknown profile: {PROFILE}")
TIMELINE = TIMELINES[PROFILE]
SMALL = "Explain AI inference briefly."
MEDIUM = ("Explain how an AI inference server handles requests, batching, "
          "token generation, and resource usage. ") * 10
EXPENSIVE = ("Explain in detail the architecture of an AI inference server, "
             "including request scheduling, KV cache, GPU execution, batching, "
             "prefill, decode, memory usage, and latency. ") * 30

_started = None
_attempt_log = None
_result_log = None
_ids = {"legit": itertools.count(1), "attacker": itertools.count(1)}


def phase_at(elapsed):
    if PROFILE == "normal":
        return "normal"
    if elapsed < TIMELINE["attack_start"]:
        return "pre_attack"
    if elapsed < TIMELINE["attack_end"]:
        return "attack"
    return "recovery"


@events.test_start.add_listener
def start_logging(environment, **kwargs):
    global _started, _attempt_log, _result_log
    host = urlparse(environment.host or "")
    if host.hostname not in {"127.0.0.1", "localhost", "::1"}:
        raise RuntimeError("This demo is restricted to your local test server.")
    RUN_DIR.mkdir(parents=True, exist_ok=True)
    random.seed(SEED)
    _started = time.time()
    _attempt_log = (RUN_DIR / "client_attempts.jsonl").open("w", buffering=65536)
    _result_log = (RUN_DIR / "client_requests.jsonl").open("w", buffering=65536)
    (RUN_DIR / "load_profile.json").write_text(json.dumps({
        "profile": PROFILE, "run_id": RUN_ID, "start_ts": _started,
        "seed": SEED, "model": MODEL, "host": environment.host,
        "generator": "Locust closed-loop virtual users", "legitimate_users": 4,
        **TIMELINE,
    }, indent=2))


@events.test_stop.add_listener
def close_logging(environment, **kwargs):
    for handle in (_attempt_log, _result_log):
        if handle is not None and not handle.closed:
            handle.close()


@events.request.add_listener
def record_request(name, response_time, response=None, context=None,
                   exception=None, **kwargs):
    if not context or not context.get("request_id") or _result_log is None:
        return
    status = getattr(response, "status_code", 0) or 0
    usage = {}
    valid = False
    if 200 <= status < 300:
        try:
            body = response.json()
            usage = body.get("usage", {})
            valid = bool(body.get("choices")) and exception is None
        except (ValueError, AttributeError):
            pass
    headers = getattr(response, "headers", {})
    row = dict(context)
    row.update({
        "end_ts": time.time(), "status": status,
        "latency_ms": response_time, "valid_response": valid,
        "prompt_tokens": usage.get("prompt_tokens"),
        "completion_tokens": usage.get("completion_tokens"),
        "gateway_reason": headers.get("X-Gateway-Reason", ""),
        "error": str(exception) if exception else "",
    })
    _result_log.write(json.dumps(row) + "\n")


class BaseTraffic(HttpUser):
    abstract = True
    traffic_label = "legit"

    def on_start(self):
        self.client_id = f"{self.traffic_label}-{next(_ids[self.traffic_label])}"
        self.sequence = 0
        self.client.trust_env = False

    def send_inference(self, prompt, max_tokens):
        self.sequence += 1
        now = time.time()
        elapsed = now - (_started or now)
        request_id = f"{RUN_ID}-{self.client_id}-{self.sequence}"
        # The nonce is deterministic per user/sequence, so paired runs use the
        # same prompt pattern. It is a documented change from the old tests.
        prompt = f"Request {self.client_id}-{self.sequence}. " + prompt
        context = {
            "request_id": request_id, "run_id": RUN_ID, "profile": PROFILE,
            "client_id": self.client_id, "traffic_label": self.traffic_label,
            "start_ts": now, "elapsed_s": elapsed, "phase": phase_at(elapsed),
            "max_tokens": max_tokens,
        }
        _attempt_log.write(json.dumps(context) + "\n")
        # No application-level retries; 429 and 503 remain real failed requests
        # in Locust, and their reasons are recorded separately for analysis.
        with self.client.post(
            "/v1/chat/completions",
            json={"model": MODEL, "messages": [{"role": "user", "content": prompt}],
                  "max_tokens": max_tokens, "temperature": 0},
            headers={"X-Client-ID": self.client_id, "X-Request-ID": request_id,
                     "X-Traffic-Label": self.traffic_label},
            context=context, name=self.traffic_label,
            timeout=(3, 120), catch_response=True,
        ) as response:
            if 200 <= response.status_code < 300:
                try:
                    if not response.json().get("choices"):
                        response.failure("2xx without a completion")
                except ValueError:
                    response.failure("2xx with invalid JSON")


class LegitimateUser(BaseTraffic):
    abstract = False
    fixed_count = 4
    traffic_label = "legit"
    wait_time = between(0.5, 1.5)

    @task
    def legitimate(self):
        self.send_inference(SMALL, 32)


class AttackerUser(BaseTraffic):
    abstract = False
    weight = 1
    traffic_label = "attacker"
    wait_time = between(1.5, 2.5) if PROFILE == "low_slow" else between(0.05, 0.15)

    @task
    def attacker(self):
        if PROFILE == "low_slow":
            self.send_inference(EXPENSIVE, 256)
        else:
            self.send_inference(MEDIUM, 128)


class DemoShape(LoadTestShape):
    def tick(self):
        t = self.get_run_time()
        if t >= TIMELINE["end"]:
            return None
        if (PROFILE == "normal" or t < TIMELINE["attack_start"]
                or t >= TIMELINE["attack_end"]):
            return (4, 20, [LegitimateUser])
        return (TIMELINE["total_users"], 20, [LegitimateUser, AttackerUser])

## B5 — Run the selected OFF/ON pairs

Both modes use the same backend, gateway hop, profile settings and seed. A unique folder per condition contains `client_attempts.jsonl`, `client_requests.jsonl`, Locust CSVs, `gateway.jsonl`, `gateway_samples.jsonl`, and configuration/timing metadata.

Run only against your own local test endpoint. Leave `PROFILES=["flood"]` for the quickest new pair. Change it to the four profile names in A1 for the full eight-condition matrix. This code does **not** run the optional VAE; ON means the two required rule-based mechanisms plus the repo's existing admission policy.

In [ ]:
LIVE_RUNS = {}
if RUN_LIVE_DEMO:
    pair_id = time.strftime("demo-%Y%m%d-%H%M%S") + "-" + uuid4().hex[:6]
    for profile in PROFILES:
        if profile not in {"normal", "spike", "flood", "low_slow"}:
            raise ValueError("Unknown profile: " + profile)
        LIVE_RUNS[profile] = {}
        for mode in ("off", "on"):
            LIVE_RUNS[profile][mode.upper()] = run_locust_profile(profile, mode, pair_id)
    (RESULTS / "latest_demo_runs.json").write_text(json.dumps({
        profile: {mode: str(path) for mode, path in runs.items()}
        for profile, runs in LIVE_RUNS.items()
    }, indent=2))
    print("Selected experiments finished. Continue to B6 and B7.")
else:
    print("Skipped new experiments. The historical result is in A2.")

## B6 — Summarize new client-side results

Calculates p50/p95 from **successful legitimate completions** and reports success/failure alongside them. Attack profiles are summarized over requests **started during the attack window**, normal over its normal window. Started requests without a completion record count as unsuccessful and are reported separately; they are not silently dropped.

Gateway latency is not substituted for client-observed latency. The summary is saved separately from the historical CSV. With small request counts, interpret p95 cautiously.

In [ ]:
def summarize_client_run(run_dir):
    attempt_path = run_dir / "client_attempts.jsonl"
    result_path = run_dir / "client_requests.jsonl"
    if not attempt_path.exists() or attempt_path.stat().st_size == 0:
        raise RuntimeError(f"No attempted requests recorded in {run_dir}")
    attempts = pd.read_json(attempt_path, lines=True, convert_dates=False)
    if result_path.exists() and result_path.stat().st_size:
        completed = pd.read_json(result_path, lines=True, convert_dates=False)
    else:
        completed = pd.DataFrame(columns=["request_id", "status", "latency_ms", "valid_response", "error"])
    if attempts["request_id"].duplicated().any() or completed["request_id"].duplicated().any():
        raise RuntimeError("Duplicate request IDs: check for mixed run logs.")
    data = attempts.merge(completed[["request_id", "status", "latency_ms", "valid_response", "error"]],
                          on="request_id", how="left", validate="one_to_one", indicator=True)
    profile = attempts["profile"].iloc[0]
    window = "normal" if profile == "normal" else "attack"
    data = data[data["phase"] == window].copy()
    rows = []
    for label in ("legit", "attacker"):
        group = data[data["traffic_label"] == label]
        if group.empty:
            continue
        status = pd.to_numeric(group["status"], errors="coerce").fillna(0)
        ok = status.between(200, 299) & group["valid_response"].fillna(False).astype(bool)
        latency = pd.to_numeric(group.loc[ok, "latency_ms"], errors="coerce").dropna() / 1000
        rows.append({
            "profile": profile, "traffic_label": label, "window": window,
            "attempted": len(group), "success_rate_pct": 100 * ok.mean(),
            "failure_rate_pct": 100 * (1 - ok.mean()),
            "rejection_rate_pct": 100 * status.isin([429, 503]).mean(),
            "transport_or_unfinished": int((status == 0).sum()),
            "http_502_504": int(status.isin([502, 504]).sum()),
            "unfinished": int((group["_merge"] == "left_only").sum()),
            "p50_s": latency.quantile(0.50) if len(latency) else float("nan"),
            "p95_s": latency.quantile(0.95) if len(latency) else float("nan"),
        })
    return rows

if LIVE_RUNS:
    rows = []
    for profile, runs in LIVE_RUNS.items():
        for mode, run_dir in runs.items():
            for row in summarize_client_run(run_dir):
                row.update(mode=mode, run_dir=str(run_dir))
                rows.append(row)
    live_summary = pd.DataFrame(rows)
    display(live_summary)
    live_summary.to_csv(RESULTS / "live_before_after.csv", index=False)
    legit = live_summary[live_summary["traffic_label"] == "legit"]
    table = legit.pivot(index="profile", columns="mode", values="p95_s")
    fig, ax = plt.subplots(figsize=(8, 4))
    table.plot.bar(ax=ax, rot=0)
    ax.set(ylabel="Successful legitimate p95 (s)", xlabel="Profile",
           title="New runs: client-observed latency (see success rates in CSV)")
    fig.tight_layout()
    fig.savefig(RESULTS / "live_legitimate_p95.png", dpi=200, bbox_inches="tight")
    plt.show()
    plt.close(fig)
else:
    print("No new runs in this session; the historical CSV is unchanged.")

## B7 — Queue chart for the new pair

Uses the same plotting function as A3, this time with the new run directories and saved load-start timestamps. Both zero or similar queue peaks are valid observations. Missing upstream telemetry is not evidence of an empty queue.

In [ ]:
if LIVE_RUNS:
    selected = "flood" if "flood" in LIVE_RUNS else next(iter(LIVE_RUNS))
    QUEUE_PATHS = {mode: run_dir / "gateway_samples.jsonl"
                   for mode, run_dir in LIVE_RUNS[selected].items()}
    queue_png = plot_queue_depth(QUEUE_PATHS, RESULTS)
else:
    print("For the historical runs, use A4.")

# C — Save the evidence before the runtime ends

This exports generated CSV/PNG files **plus the selected raw run folders** into `results/demo_evidence.zip`. The original `.gitignore` excludes `runs/`, so keeping only a GitHub notebook does not preserve those raw files. The ZIP under `results/` solves that without changing `.gitignore`.

The original notebooks remain in the project for provenance. Do not label historical outputs or synthetic tests as a newly executed real-GPU run. VAE training/results are outside this demo's required two-mechanism comparison.

In [ ]:
import zipfile

bundle = RESULTS / "demo_evidence.zip"
selected_dirs = {path.parent for path in QUEUE_PATHS.values() if path.is_file()}
for runs in LIVE_RUNS.values():
    selected_dirs.update(runs.values())
with zipfile.ZipFile(bundle, "w", zipfile.ZIP_DEFLATED) as z:
    for file in RESULTS.iterdir():
        if file.is_file() and file != bundle and file.suffix.lower() in {".png", ".csv", ".json", ".txt"}:
            z.write(file, file.relative_to(REPO))
    for directory in sorted(selected_dirs):
        for file in directory.rglob("*"):
            if file.is_file():
                try:
                    name = file.relative_to(REPO)
                except ValueError:
                    name = Path("runs") / directory.name / file.relative_to(directory)
                z.write(file, name)
print("Saved:", bundle)
print("Included raw run folders:", len(selected_dirs))
print("A zero count means the raw logs still need to be recovered from the original runtime.")

### Optional download cell

Run manually in Colab after the evidence archive has been created. Add `final_demo.ipynb`, `loadgen/locustfile.py`, and the evidence ZIP/plots to your submission. Never include tokens or credentials.

In [ ]:
# Uncomment to download from Colab:
# from google.colab import files
# files.download(str(RESULTS / "demo_evidence.zip"))
# files.download(str(RESULTS / "queue_depth_off_vs_on.png"))

## Provenance and deliberate changes

- `integration.ipynb`: successful real-GPU installation/launch settings, gateway configuration, successful historical smoke results and existing CSV/PNG.
- `testing.ipynb`: concurrency probe and the earlier sampler idea. Its saved failed probe is not relabeled a successful run.
- Four Locust profiles: restored from the earlier conversation because they are absent from the uploaded testing notebook.
- Added: bounded process waits, one managed gateway, unique run IDs, warmup/forwarding checks, client IDs/labels, raw request logging, raw-log queue plotting, and artifact export.
- Not changed: gateway admission, rate-limiting, VAE, fake-backend or logging implementation; historical source notebooks/CSV/PNG.
- Validation of this deliverable covers notebook/code structure and local checks, not a fresh Colab/T4 benchmark. Replayed performance numbers must come from your own run.